# Chapitre 3 - Cours 3 : LAR, PCR, PLS et comparaison des méthodes

Ce dernier notebook de cours couvre les sections 3.4.4 à 3.6 du livre :

1. la **régression du moindre angle** (LAR, *least angle regression*), son lien avec le
   lasso et ses degrés de liberté (figures 3.14, 3.15) ;
2. les méthodes à **directions dérivées** : régression sur composantes principales (PCR)
   et moindres carrés partiels (PLS) (figure 3.17, algorithme 3.3) ;
3. la **comparaison** de toutes les méthodes du chapitre sur la prostate (figure 3.7
   complète, tableau 3.3) et sur un problème jouet à deux variables (figure 3.18).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import lars_path
from esl import ch03_regression as cr

pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)


def show(df):
    """Affiche un tableau arrondi, les coefficients absents (NaN) laissés en blanc."""
    return df.round(3).astype(object).where(df.notna(), "")

In [2]:
Xtr, ytr, Xte, yte, names = cr.load_prostate_standardized()
N, p = Xtr.shape
fit_ls = cr.ols_fit(Xtr, ytr)


def test_error(b0, beta):
    """Erreur quadratique moyenne sur l'échantillon de test."""
    return np.mean((yte - b0 - Xte @ beta) ** 2)

## 1. Régression du moindre angle (LAR)

LAR est une version « démocratique » de la sélection pas à pas ascendante. Au lieu
d'ajouter **entièrement** une variable au modèle, elle n'en ajoute que « la part
méritée » (algorithme 3.2 du livre) :

1. Standardiser les prédicteurs (moyenne nulle, norme 1). Partir de $\mathbf{r} = \mathbf{y} - \bar y$
   et $\beta_1 = \dots = \beta_p = 0$.
2. Trouver le prédicteur $\mathbf{x}_j$ le plus corrélé à $\mathbf{r}$.
3. Faire croître $\beta_j$ de 0 vers son coefficient des moindres carrés
   $\langle\mathbf{x}_j, \mathbf{r}\rangle$, **jusqu'à ce qu'un autre** prédicteur
   $\mathbf{x}_k$ ait autant de corrélation avec le résidu courant.

4. Faire évoluer $\beta_j$ et $\beta_k$ dans la direction de leur coefficient des moindres
   carrés **joint** (régression du résidu sur $(\mathbf{x}_j, \mathbf{x}_k)$), jusqu'à ce
   qu'un troisième prédicteur $\mathbf{x}_l$ rattrape la même corrélation.
5. Continuer ainsi jusqu'à ce que tous les prédicteurs soient entrés. Après
   $\min(N-1, p)$ étapes, on obtient la solution des moindres carrés complète.

**La direction de l'étape $k$.** Notons $\mathcal{A}_k$ l'ensemble **actif** au début de
l'étape $k$, $\mathbf{X}_{\mathcal{A}_k}$ les colonnes correspondantes et
$\mathbf{r}_k = \mathbf{y} - \mathbf{X}_{\mathcal{A}_k}\beta_{\mathcal{A}_k}$ le résidu.
La direction des coefficients est (éq. (3.55))

$$
\delta_k = (\mathbf{X}_{\mathcal{A}_k}^T\mathbf{X}_{\mathcal{A}_k})^{-1}\mathbf{X}_{\mathcal{A}_k}^T\mathbf{r}_k ,
$$

et l'on pose $\beta_{\mathcal{A}_k}(\alpha) = \beta_{\mathcal{A}_k} + \alpha\,\delta_k$. Les
valeurs ajustées évoluent selon $\hat{\mathbf{f}}_k(\alpha) = \hat{\mathbf{f}}_k + \alpha\,\mathbf{u}_k$
avec $\mathbf{u}_k = \mathbf{X}_{\mathcal{A}_k}\delta_k$.

**Pourquoi les corrélations restent à égalité.** Supposons qu'au début de l'étape toutes
les variables actives aient la même corrélation absolue $C$ avec le résidu :
$\mathbf{X}_{\mathcal{A}}^T\mathbf{r}_k = C\,\mathbf{s}$ où $\mathbf{s}$ est le vecteur
des signes. Alors $\mathbf{X}_{\mathcal{A}}^T\mathbf{u}_k = \mathbf{X}_{\mathcal{A}}^T\mathbf{X}_{\mathcal{A}}\delta_k = \mathbf{X}_{\mathcal{A}}^T\mathbf{r}_k = C\,\mathbf{s}$, donc

$$
\mathbf{X}_{\mathcal{A}}^T\bigl(\mathbf{r}_k - \alpha\,\mathbf{u}_k\bigr) = (1 - \alpha)\,C\,\mathbf{s} :
$$

les corrélations actives décroissent **ensemble**, linéairement, et restent égales en
valeur absolue. Comme $\mathbf{u}_k$ fait le même produit scalaire avec chaque
$s_j\mathbf{x}_j$ (de norme 1), il fait **le même angle** avec chacune : c'est l'origine du
nom « moindre angle » (exercice 3.24).